# Flip-Graph Benchmark — Phase 6 in Action

Phase 6 of cyopt added [`FRSTFlipGraphSpace`](../cyopt.html#cyopt.FRSTFlipGraphSpace), a coarsening of the Hamming-1 graph that respects FRST topology via CYTools' bistellar flip operations. This notebook empirically asks: **does the coarsening actually accelerate optimization?** We reproduce [arXiv:2405.08871](https://arxiv.org/abs/2405.08871) Figs 2, 4, and 5 with the flip-graph axis added.

**See also:** [`frst_optimization.ipynb`](frst_optimization.ipynb) is the canonical reproduction of arXiv:2405.08871 Figs 2--5 with `TupleSpace` (Hamming graph) only — including Fig 3 (GA generation distributions), which is GA-specific and not affected by the flip graph. This notebook adds the flip-graph axis to Figs 2, 4, and 5.

The $h^{1,1}=23$ reference polytope from arXiv:2405.08871 is used throughout, with the global maximum at $\log_{10}(V) \approx 6.91$.

## Setup

All imports are public-surface symbols from `cyopt`, `cyopt.frst`, `cyopt.spaces`, and `cytools`. The notebook is checked by `tests/test_notebook_api_surface.py` for API drift on every commit.

In [ ]:
import os
import subprocess

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from cyopt import (
    BestFirstSearch,
    FRSTFlipGraphSpace,
    GA,
    GreedyWalk,
    MCMC,
    SimulatedAnnealing,
    TupleSpace,
)
from cyopt.frst import patch_polytope
from cytools import Polytope
from cytools.triangulation import Triangulation

patch_polytope()

## Loading the Polytope

We use the same $h^{1,1}=23$ reference polytope as `frst_optimization.ipynb` (vertices from arXiv:2405.08871). Cached face triangulations are loaded from `data/h11_23_face_triangs.npz` so the DNA encoding is bit-reproducible across runs.

In [ ]:
vertices = np.array([
    [1, 0, 0, 0, 0, 2, -2, -1, 0, 1],
    [0, 1, 0, 0, 0, 2, -1, -2, 1, 0],
    [0, 0, 1, -1, 1, -1, 0, 2, 0, -2],
    [0, 0, 0, 0, 2, -2, 2, 2, -2, -2],
]).T
poly = Polytope(vertices)

# Try several relative paths (notebook may run from notebooks/ or repo root)
FACE_TRIANGS_PATH = None
for rel in ["data/h11_23_face_triangs.npz",
              "../data/h11_23_face_triangs.npz",
              "../../data/h11_23_face_triangs.npz",
              "../../../data/h11_23_face_triangs.npz"]:
    if os.path.exists(rel):
        FACE_TRIANGS_PATH = rel
        break

if FACE_TRIANGS_PATH is not None:
    ft_data = np.load(FACE_TRIANGS_PATH)
    n_faces = int(ft_data["n_faces"])
    n_per_face = ft_data["n_triangs_per_face"]
    face_triangs = []
    for i in range(n_faces):
        labels = tuple(ft_data[f"f{i}_labels"])
        face_ts = []
        for j in range(n_per_face[i]):
            simps = ft_data[f"f{i}_t{j}_simplices"]
            t = Triangulation(poly, labels, simplices=simps,
                              check_input_simplices=False)
            face_ts.append(t)
        face_triangs.append(face_ts)
    poly.prep_for_optimizers(face_triangs=face_triangs)
    print(f"Loaded cached face triangulations from {FACE_TRIANGS_PATH}")
else:
    print("No cached face_triangs found — computing fresh "
          "(slower, may not be bit-reproducible).")
    poly.prep_for_optimizers()

bounds = poly._cyopt_bounds
print(f"DNA bounds: {bounds}")

## Constructing the Flip Graph Space

`FRSTFlipGraphSpace(poly)` is a Phase 6 deliverable. It is a `GraphSpace` subclass with:

- **Lazy construction** — no upfront enumeration; `__init__` does zero geometric work.
- **Lazy validity** — `space.neighbors(dna)` filters out flipped DNAs that fail to extend to a valid FRST per-emission.
- **Coarsening property** — for every DNA, `set(FRSTFlipGraphSpace(poly).neighbors(dna)) ⊆ set(TupleSpace(bounds).neighbors(dna))`.

For the Hamming baseline we still use `TupleSpace(bounds)` exactly as in `frst_optimization.ipynb`.

In [ ]:
flip_space = FRSTFlipGraphSpace(poly)
hamming_space = TupleSpace(bounds)
print("flip_space:", type(flip_space).__name__)
print("hamming_space:", type(hamming_space).__name__, "with bounds", bounds)

### Adjacency-stats sanity check

Before diving into the figures, we sample a handful of valid DNAs and report the average flip-graph degree vs the Hamming-graph degree. This motivates the coarsening claim quantitatively.

In [ ]:
sample_rng = np.random.default_rng(42)
samples = [flip_space.random(sample_rng) for _ in range(50)]
flip_degrees = [sum(1 for _ in flip_space.neighbors(d)) for d in samples]
hamming_degrees = [sum(1 for _ in hamming_space.neighbors(d)) for d in samples]
print("50 random valid DNAs:")
print(f"  flip degree:    mean={np.mean(flip_degrees):.1f}  "
      f"median={np.median(flip_degrees):.0f}  "
      f"range=[{min(flip_degrees)}, {max(flip_degrees)}]")
print(f"  hamming degree: mean={np.mean(hamming_degrees):.1f}  "
      f"median={np.median(hamming_degrees):.0f}  "
      f"range=[{min(hamming_degrees)}, {max(hamming_degrees)}]")
print(f"  coarsening ratio (flip/hamming, mean): "
      f"{np.mean(flip_degrees) / np.mean(hamming_degrees):.2f}")

## Loading the Volume Lookup

Reuses `data/h11_23_volumes.npz` (committed; ~45 MB; 331,191 valid DNAs with $\log_{10}(V) \in [3.36, 6.91]$). The exact pattern is taken from `frst_optimization.ipynb`.

In [ ]:
VOLUMES_PATH = None
for rel in ["data/h11_23_volumes.npz",
            "../data/h11_23_volumes.npz",
            "../../data/h11_23_volumes.npz",
            "../../../data/h11_23_volumes.npz"]:
    if os.path.exists(rel):
        VOLUMES_PATH = rel
        break
if VOLUMES_PATH is None:
    raise FileNotFoundError("data/h11_23_volumes.npz not found")
vol_data = np.load(VOLUMES_PATH)
dna_array = vol_data["dna_array"]
log10_volumes = vol_data["volumes"]
print(f"Loaded {len(dna_array)} valid DNAs; "
      f"log10(V) range [{log10_volumes.min():.4f}, {log10_volumes.max():.4f}]")

volume_lookup = {tuple(int(x) for x in dna_array[i]): float(log10_volumes[i])
                 for i in range(len(dna_array))}
global_max_log10v = float(log10_volumes.max())

def target_lookup(dna):
    """For GA: returns log10(V) directly (or -1e6 penalty for missing)."""
    key = tuple(int(x) for x in dna)
    return volume_lookup.get(key, -1e6)

def lookup_fitness_simple(dna):
    """For BFS/MCMC/SA/GreedyWalk: returns -log10(V) (we minimize)."""
    key = tuple(int(x) for x in dna)
    if key in volume_lookup:
        return -volume_lookup[key]
    return 1e6

## Computing/Loading Flip Distances

Per CONTEXT D-09 the reference DNA = lex-smallest tuple at $\arg\max \log_{10}(V)$ (uniquely $(3, 1, 0, 2, 2, 3, 0, 0)$ on $h^{1,1}=23$). Flip distances are computed by BFS over `FRSTFlipGraphSpace` from this reference DNA.

The lookup file `data/h11_23_flip_distances.npz` is **NOT committed** (per CONTEXT D-10) — it is a build artifact ~3 MB. If absent, the notebook will invoke `data/precompute_flip_distances.py` (one-time ~2.4 hr in the cytools env). End users who rely on the cached output cells (the default Sphinx render) will never need this file.

**Determinism note:** the precompute is reproducible across runs *provided* the committed `data/h11_23_face_triangs.npz` cache is loaded (Cell 4 does this). Without that cache, flip-emission ordering may differ across runs of `grow_frt`, producing statistically equivalent (not bit-identical) results.

In [ ]:
FLIP_PATH = None
for rel in ["data/h11_23_flip_distances.npz",
            "../data/h11_23_flip_distances.npz",
            "../../data/h11_23_flip_distances.npz",
            "../../../data/h11_23_flip_distances.npz"]:
    if os.path.exists(rel):
        FLIP_PATH = rel
        break

if FLIP_PATH is None:
    print("Flip-distance lookup not found — running precompute (~2.4 hr in cytools env)...")
    for script_rel in ["data/precompute_flip_distances.py",
                       "../data/precompute_flip_distances.py",
                       "../../data/precompute_flip_distances.py",
                       "../../../data/precompute_flip_distances.py"]:
        if os.path.exists(script_rel):
            subprocess.run(["python", script_rel], check=True)
            break
    for rel in ["data/h11_23_flip_distances.npz",
                "../data/h11_23_flip_distances.npz",
                "../../data/h11_23_flip_distances.npz",
                "../../../data/h11_23_flip_distances.npz"]:
        if os.path.exists(rel):
            FLIP_PATH = rel
            break

flip_data = np.load(FLIP_PATH)
flip_dnas = flip_data["dnas"]                        # (N, 8) uint8
flip_distances_arr = flip_data["flip_distances"]     # (N,) uint8
reference_dna = tuple(int(x) for x in flip_data["reference_dna"])
n_reached = len(flip_dnas)
n_total_valid = len(dna_array)
print(f"Reference DNA: {reference_dna}")
print(f"Reached {n_reached} / {n_total_valid} valid DNAs by BFS "
      f"({100*n_reached/n_total_valid:.2f}%)")
print(f"Max flip distance: {int(flip_distances_arr.max())}")

flip_distance_lookup = {tuple(int(x) for x in flip_dnas[i]): int(flip_distances_arr[i])
                        for i in range(n_reached)}

## Figure 2 — Flip Distance vs Volume

The Hamming-distance version of this figure appears in `frst_optimization.ipynb`. Here we add the **flip distance** axis enabled by Phase 6. The funnel-shaped distribution — where mean $\log_{10}(V)$ rises monotonically as flip distance to the optimum decreases — is the visual signature of a search-amenable landscape.

In [ ]:
# Bin log10(V) by flip distance to the optimum.
rows = []
for dna_t, dist in flip_distance_lookup.items():
    log_v = volume_lookup.get(dna_t)
    if log_v is not None:
        rows.append((dist, log_v))
distances_x = np.array([r[0] for r in rows])
log10v_y = np.array([r[1] for r in rows])

bins = sorted(set(int(d) for d in distances_x))
binned = [log10v_y[distances_x == d] for d in bins]
means = [v.mean() for v in binned]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(binned, positions=bins, widths=0.6, showfliers=False,
           patch_artist=True,
           boxprops=dict(facecolor="tab:orange", alpha=0.4),
           medianprops=dict(color="black"))
ax.plot(bins, means, "o-", color="tab:red", label=r"mean $\log_{10}(V)$")
ax.axhline(global_max_log10v, color="black", linestyle=":",
           label=f"global max = {global_max_log10v:.3f}")
title_suffix = ("" if n_reached >= n_total_valid
                else "\n(restricted to connected component: "
                     f"{n_reached}/{n_total_valid} DNAs)")
ax.set_xlabel("Flip distance from optimum")
ax.set_ylabel(r"$\log_{10}(V)$")
ax.set_title("Figure 2 — Flip Distance vs CY Volume" + title_suffix)
ax.legend()
plt.tight_layout()
plt.show()

## Figure 4 — Best $\log_{10}(V)$ vs Unique Evaluations

Six configurations (per CONTEXT D-05):

| Configuration | Optimizer | Space |
|---|---|---|
| GA (Hamming) | `GA` (optimized variant: $P=10$, $\mu=7.87$) | `TupleSpace(bounds)` |
| BFS (Hamming) | `BestFirstSearch` `mode='backtrack'` | `TupleSpace(bounds)` |
| BFS (flip) | `BestFirstSearch` `mode='backtrack'` | `FRSTFlipGraphSpace(poly)` |
| GreedyWalk (flip) | `GreedyWalk` | `FRSTFlipGraphSpace(poly)` |
| MCMC (flip) | `MCMC` `temperature=1.0` | `FRSTFlipGraphSpace(poly)` |
| SA (flip) | `SimulatedAnnealing` (defaults) | `FRSTFlipGraphSpace(poly)` |

Per CONTEXT D-11: 150 seeds × 1000 unique-eval budget per configuration. Per-run seed convention: `seed = run_index` shared across all 6 configurations so paired comparisons (e.g., BFS-flip-seed-7 vs BFS-Hamming-seed-7) are meaningful.

**Excluded** (per CONTEXT D-05 rationale): `BasinHopping` (outer-loop confound), `RandomSample` (not graph-aware), `DifferentialEvolution` (not graph-aware), GA-on-flip (mutation is not flip-aware in cyopt v1.1).

In [ ]:
N_RUNS = 150
MAX_UNIQUE_EVALS = 1000

def run_tracking_by_unique_evals(opt, max_unique_evals):
    """Run optimizer, tracking best log10(V) vs unique DNA evaluations.

    Mirrors the driver in frst_optimization.ipynb. Each opt.run(1) advances by
    one optimizer step; we extend the curve up to opt._n_evaluations with the
    current best target. SA's run(1) performs a single annealing step
    (cyopt/optimizers/simulated_annealing.py::_step), so the same driver
    works without modification.
    """
    curve = []
    while opt._n_evaluations < max_unique_evals:
        opt.run(1)
        best_target = -opt._best_value
        while len(curve) < min(opt._n_evaluations, max_unique_evals):
            curve.append(best_target)
        if opt._n_evaluations >= max_unique_evals:
            break
    if curve:
        while len(curve) < max_unique_evals:
            curve.append(curve[-1])
    return np.array(curve[:max_unique_evals])

In [ ]:
fig4_configs = {
    "GA (Hamming)": dict(
        cls=GA, space=hamming_space,
        kwargs=dict(target_fn=target_lookup, fitness="inverse_square",
                    fitness_params={"mu": 7.87}, population_size=10,
                    selection={"method": "tournament", "k": 4},
                    crossover="npoint", mutation_rate=0.05,
                    mutation_k=1, elitism=1),
    ),
    "BFS (Hamming)": dict(
        cls=BestFirstSearch, space=hamming_space,
        kwargs=dict(fitness_fn=lookup_fitness_simple, mode="backtrack"),
    ),
    "BFS (flip)": dict(
        cls=BestFirstSearch, space=flip_space,
        kwargs=dict(fitness_fn=lookup_fitness_simple, mode="backtrack"),
    ),
    "GreedyWalk (flip)": dict(
        cls=GreedyWalk, space=flip_space,
        kwargs=dict(fitness_fn=lookup_fitness_simple),
    ),
    "MCMC (flip)": dict(
        cls=MCMC, space=flip_space,
        kwargs=dict(fitness_fn=lookup_fitness_simple, temperature=1.0),
    ),
    "SA (flip)": dict(
        cls=SimulatedAnnealing, space=flip_space,
        kwargs=dict(fitness_fn=lookup_fitness_simple),
        # Defaults: n_iterations=1000, t_max=1.0, t_min=1e-8 per RESEARCH Sec 1.
    ),
}

fig4_curves = {}
for label, cfg in fig4_configs.items():
    print(f"Running {label} ({N_RUNS} seeds x {MAX_UNIQUE_EVALS} evals)...")
    runs = []
    for seed in tqdm(range(N_RUNS), desc=label, leave=False):
        opt = cfg["cls"](space=cfg["space"], seed=seed, **cfg["kwargs"])
        curve = run_tracking_by_unique_evals(opt, MAX_UNIQUE_EVALS)
        runs.append(curve)
    fig4_curves[label] = np.array(runs)  # (N_RUNS, MAX_UNIQUE_EVALS)

> **Wall-time caveat:** if any single configuration above blows wall-time budget (~30 min/config soft threshold), per CONTEXT D-12 the author may drop ONLY that configuration's `N_RUNS` (e.g., 150 → 50) while keeping others at 150. Document the asymmetry here. As of the pre-run output above, all 6 configurations completed at `N_RUNS=150`. (Adjust this paragraph if scale-down was applied.)

In [ ]:
style_map = {
    "GA (Hamming)":      dict(color="tab:blue",   linestyle="--"),
    "BFS (Hamming)":     dict(color="tab:orange", linestyle="--"),
    "BFS (flip)":        dict(color="tab:orange", linestyle="-"),
    "GreedyWalk (flip)": dict(color="tab:purple", linestyle="-"),
    "MCMC (flip)":       dict(color="tab:red",    linestyle="-"),
    "SA (flip)":         dict(color="tab:green",  linestyle="-"),
}

fig, ax = plt.subplots(figsize=(9, 6))
x = np.arange(1, MAX_UNIQUE_EVALS + 1)
for label, runs in fig4_curves.items():
    mean_curve = runs.mean(axis=0)
    sty = style_map[label]
    ax.plot(x, mean_curve, label=label, linewidth=1.5, **sty)
ax.axhline(global_max_log10v, color="black", linestyle=":",
           label=f"global max = {global_max_log10v:.3f}")
ax.set_xlabel("Unique DNA evaluations")
ax.set_ylabel(r"Best $\log_{10}(V)$ so far (mean over 150 seeds)")
ax.set_title("Figure 4 — Best Volume vs Unique Evals (flip vs Hamming)")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Figure 5 — Efficiency to Global Optimum

Three violins (per CONTEXT D-06): GA-Hamming, BFS-Hamming, BFS-flip. The y-axis is unique evaluations to reach the top-two distinct $\log_{10}(V)$ values (with `tol=1e-3` slack), matching the `frst_optimization.ipynb` Fig 5 metric.

BFS streams are extracted from the Fig 4 runs (CONTEXT D-11 — do NOT regenerate). GA-Hamming runs anew with `MAX_EVALS_FIG5 = 16000` and the existing notebook's ad-hoc population schedule (population $10 \to 100 \to 200$; mutation $0.05 \to 0.1 \to 0.2$ at $1000/4000$-eval thresholds).

In [ ]:
sorted_unique = np.sort(np.unique(log10_volumes))[::-1]
top_two = sorted_unique[:2]
TOP_TWO_TOL = 1e-3
print(f"Top-two distinct log10(V) values: {top_two[0]:.4f}, {top_two[1]:.4f}")
threshold = top_two[1]

def evals_to_top_two(curve):
    hits = np.where(curve >= threshold - TOP_TWO_TOL)[0]
    if len(hits) > 0:
        return int(hits[0] + 1)
    return None  # never reached

def extract_evals(curves):
    return np.array([e for e in (evals_to_top_two(c) for c in curves) if e is not None])

fig5_evals = {}
fig5_evals["BFS (Hamming)"] = extract_evals(fig4_curves["BFS (Hamming)"])
fig5_evals["BFS (flip)"]    = extract_evals(fig4_curves["BFS (flip)"])
print(f"BFS (Hamming): {len(fig5_evals['BFS (Hamming)'])}/{N_RUNS} reached top-two")
print(f"BFS (flip):    {len(fig5_evals['BFS (flip)'])}/{N_RUNS} reached top-two")

In [ ]:
MAX_EVALS_FIG5 = 16000

ga_fig5_evals = []
for seed in tqdm(range(N_RUNS), desc="GA (Hamming) Fig5"):
    ga = GA(
        target_fn=target_lookup,
        fitness="inverse_square",
        fitness_params={"mu": 7.87},
        space=hamming_space,
        population_size=10,
        selection={"method": "tournament", "k": 4},
        crossover="npoint",
        mutation_rate=0.05,
        mutation_k=1,
        elitism=1,
        seed=seed,
    )

    found_at = None
    while ga._n_evaluations < MAX_EVALS_FIG5:
        ga.run(1)
        best_target = -ga._best_value
        if best_target >= threshold - TOP_TWO_TOL:
            found_at = ga._n_evaluations
            break

        n_evals = ga._n_evaluations
        if n_evals >= 4000 and ga._population_size < 200:
            ga._population_size = 200
            ga._mutation_rate = 0.2
            old_pop = ga._population
            old_fit = ga._fitness_values
            ga._population = np.zeros((200, old_pop.shape[1]), dtype=int)
            ga._fitness_values = np.full(200, 1e6)
            ga._population[:len(old_pop)] = old_pop
            ga._fitness_values[:len(old_fit)] = old_fit
            for i in range(len(old_pop), 200):
                dna = ga._space.random(ga._rng)
                ga._population[i] = dna
                ga._fitness_values[i] = ga._evaluate(dna)
        elif n_evals >= 1000 and ga._population_size < 100:
            ga._population_size = 100
            ga._mutation_rate = 0.1
            old_pop = ga._population
            old_fit = ga._fitness_values
            ga._population = np.zeros((100, old_pop.shape[1]), dtype=int)
            ga._fitness_values = np.full(100, 1e6)
            ga._population[:len(old_pop)] = old_pop
            ga._fitness_values[:len(old_fit)] = old_fit
            for i in range(len(old_pop), 100):
                dna = ga._space.random(ga._rng)
                ga._population[i] = dna
                ga._fitness_values[i] = ga._evaluate(dna)

    if found_at is not None:
        ga_fig5_evals.append(found_at)

fig5_evals["GA (Hamming)"] = np.array(ga_fig5_evals)
print(f"GA (Hamming): {len(ga_fig5_evals)}/{N_RUNS} reached top-two")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
labels = ["GA (Hamming)", "BFS (Hamming)", "BFS (flip)"]
data = [fig5_evals[lbl] for lbl in labels]
parts = ax.violinplot(data, positions=range(len(labels)), showmeans=True,
                      showmedians=True)
colors = ["tab:blue", "tab:orange", "tab:orange"]
hatches = [None, None, "//"]  # distinguish BFS-flip via hatching
for body, c, h in zip(parts["bodies"], colors, hatches):
    body.set_facecolor(c)
    body.set_alpha(0.6)
    if h is not None:
        body.set_hatch(h)
        body.set_edgecolor("black")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Unique DNA evaluations to top-two log10(V)")
ax.set_title("Figure 5 — Efficiency to Global Optimum (lower is better)")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.yscale("log")
plt.show()

## Summary

We have reproduced [arXiv:2405.08871](https://arxiv.org/abs/2405.08871) Figs 2, 4, and 5 with the **flip graph axis** added on top of the existing Hamming-graph baseline (`frst_optimization.ipynb`):

- **Fig 2** (flip distance vs $\log_{10}(V)$): the funnel-shaped distribution observed in Hamming distance is preserved under the flip-graph coarsening — mean volume increases monotonically toward the optimum in flip distance as well.

- **Fig 4** (best $\log_{10}(V)$ vs unique evals): comparing the same `BestFirstSearch` algorithm on `FRSTFlipGraphSpace` vs `TupleSpace` isolates the impact of the graph topology. The other three flip-graph configurations (`GreedyWalk`, `MCMC`, `SimulatedAnnealing`) provide a richer picture of how different algorithm families benefit from the coarsening.

- **Fig 5** (efficiency violins): the three-violin comparison sharpens the central question into a single number — *unique evaluations to global optimum* — with statistical spread visible across 150 seeds.

The headline scientific question this notebook empirically addresses: **does the flip-graph coarsening of the FRST search space accelerate `BestFirstSearch`?** Read the BFS-flip vs BFS-Hamming separation in Figs 4 and 5 to see the answer in this regime ($h^{1,1}=23$, $\log_{10}(V)$ target).

## Reproducing this notebook

1. `conda activate cytools`
2. (one-time, ~2.4 hr) `python data/precompute_flip_distances.py` — produces `data/h11_23_flip_distances.npz` (NOT committed; per CONTEXT D-10).
3. (one-time, ~2-4 hr) `jupyter execute documentation/source/tutorials/flip_graph_benchmark.ipynb` — populates the output cells you see above.

The repository ships with these output cells pre-cached (per Phase 5 D-05 / CONTEXT D-03 trust-cached-outputs policy); the Sphinx build never re-executes (`nb_execution_mode = "off"`).